# Preprocesamiento de 5 datasets de la PNDA

Notebook para Google Colab. Descarga los 5 datasets directamente desde la Plataforma Nacional de Datos Abiertos y aplica el preprocesamiento descrito en el informe.

**Datasets**:
1. ETES - Gasto presupuestal de Entidades de Tratamiento Empresarial (MEF) 2024
2. Fallecidos por COVID-19 (MINSA)
3. ENDES 2024 - REC42 (INEI)
4. ENAHO 2024 - Modulo 01 (INEI)
5. NNA atendidos en CEDIF (INABIF)

**Como ejecutarlo en Colab**: subir este archivo a https://colab.research.google.com/ (Archivo > Subir notebook) y correr las celdas en orden.

In [ ]:
# Dependencias (Colab ya las trae casi todas)
!pip install -q pandas numpy pyarrow

In [ ]:
import os
import urllib.request

os.makedirs("datasets/etes", exist_ok=True)
os.makedirs("datasets/covid_fallecidos", exist_ok=True)
os.makedirs("datasets/endes_2024", exist_ok=True)
os.makedirs("datasets/enaho_2024", exist_ok=True)
os.makedirs("datasets/cuidado_diurno_nna", exist_ok=True)

UA = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"

def descargar(url, destino):
    if os.path.exists(destino) and os.path.getsize(destino) > 1000:
        print(f"  ya existe: {destino}")
        return
    req = urllib.request.Request(url, headers={"User-Agent": UA, "Accept": "text/csv,*/*"})
    print(f"  descargando {os.path.basename(destino)}...")
    with urllib.request.urlopen(req, timeout=600) as r, open(destino, "wb") as f:
        f.write(r.read())
    print(f"  -> {os.path.getsize(destino)/1024/1024:.2f} MB")

URLS = {
    "datasets/etes/2024-Gastos-ETES.csv": "https://fs.datosabiertos.mef.gob.pe/datastorefiles/2024-Gastos-ETES.csv",
    "datasets/covid_fallecidos/fallecidos_covid.csv": "https://files.minsa.gob.pe/s/t9AFqRbXw3F55Ho/download",
    "datasets/endes_2024/REC42_2024.csv": "https://www.datosabiertos.gob.pe/sites/default/files/REC42_2024.csv",
    "datasets/enaho_2024/Enaho01-2024-100.csv": "https://www.datosabiertos.gob.pe/sites/default/files/Enaho01-2024-100.csv",
    "datasets/cuidado_diurno_nna/NNA_Octubre_2025.csv": "https://www.datosabiertos.gob.pe/sites/default/files/NNA%20atendidos%20en%20CEDIF%20Octubre%202025.csv",
    "datasets/cuidado_diurno_nna/NNA_Noviembre_2025.csv": "https://www.datosabiertos.gob.pe/sites/default/files/NNA%20atendidos%20en%20CEDIF%20Noviembre%202025.csv",
    "datasets/cuidado_diurno_nna/NNA_Diciembre_2025.csv": "https://www.datosabiertos.gob.pe/sites/default/files/NNA%20atendidos%20en%20CEDIF%20Diciembre%202025.csv",
    "datasets/cuidado_diurno_nna/NNA_Febrero_2026.csv": "https://www.datosabiertos.gob.pe/sites/default/files/NNA%20atendidos%20en%20CEDIF%20Febrero%202026.csv",
}

for destino, url in URLS.items():
    descargar(url, destino)

print("\nListo. Archivos descargados.")


## 1. ETES - Gasto Presupuestal ETES 2024

In [ ]:
import pandas as pd

etes_raw = pd.read_csv("datasets/etes/2024-Gastos-ETES.csv", encoding="utf-8", low_memory=False)
print("ANTES:", etes_raw.shape, "|", round(etes_raw.memory_usage(deep=True).sum()/1024/1024, 1), "MB")
etes_raw.head(3)

In [ ]:
etes = etes_raw.copy()

cols_codigo = [c for c in etes.columns if (c + "_NOMBRE") in etes.columns]
etes = etes.drop(columns=cols_codigo)

key = ["SEC_EJEC", "SEC_FUNC", "META", "FINALIDAD"]
ejec = (etes[etes["MES_EJE"].between(1, 12)]
        .groupby(key, dropna=False)["MONTO_EJECUCION"].sum()
        .reset_index().rename(columns={"MONTO_EJECUCION": "EJEC_ANUAL"}))
pres = etes[(etes["MES_EJE"] == 0) & (etes["MONTO_PIM"] > 0)]
etes = pres.merge(ejec, on=key, how="left").fillna({"EJEC_ANUAL": 0})
etes["PCT_EJECUCION"] = (etes["EJEC_ANUAL"] / etes["MONTO_PIM"] * 100).round(2)
etes = etes.drop(columns=["MES_EJE", "MONTO_EJECUCION"], errors="ignore")

for c in ["GRUPO_ENTIDAD_NOMBRE", "DEPARTAMENTO_EJECUTORA_NOMBRE", "FUNCION_NOMBRE"]:
    if c in etes.columns:
        etes[c] = etes[c].astype("category")

print("DESPUES:", etes.shape, "|", round(etes.memory_usage(deep=True).sum()/1024/1024, 1), "MB")
etes.head(3)

## 2. Fallecidos COVID-19

In [ ]:
import numpy as np
import unicodedata

fall_raw = pd.read_csv("datasets/covid_fallecidos/fallecidos_covid.csv", sep=";", encoding="utf-8", low_memory=False)
print("ANTES:", fall_raw.shape, "| dups:", fall_raw.duplicated().sum())

def quitar_tildes(s):
    if pd.isna(s):
        return s
    s = str(s).upper().strip()
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

fall = fall_raw.copy()
for c in ["FECHA_FALLECIMIENTO", "FECHA_CORTE"]:
    fall[c] = pd.to_datetime(fall[c].astype(str), format="%Y%m%d", errors="coerce")
fall["UBIGEO"] = fall["UBIGEO"].apply(lambda x: str(int(x)).zfill(6) if pd.notna(x) else np.nan)
fall = fall.drop_duplicates()
fall = fall[fall["EDAD_DECLARADA"].between(0, 110)]
for c in ["DEPARTAMENTO", "PROVINCIA", "DISTRITO"]:
    fall[c] = fall[c].apply(quitar_tildes)
fall = fall.dropna(subset=["DEPARTAMENTO", "PROVINCIA", "UBIGEO"])
fall["EDAD_DECLARADA"] = fall["EDAD_DECLARADA"].astype("int8")
fall["SEXO"] = fall["SEXO"].astype("category")
fall["DEPARTAMENTO"] = fall["DEPARTAMENTO"].astype("category")
fall["CLASIFICACION_DEF"] = fall["CLASIFICACION_DEF"].astype("category")

print("DESPUES:", fall.shape, "|", round(fall.memory_usage(deep=True).sum()/1024/1024, 1), "MB")
fall.head(3)

## 3. ENDES 2024 - REC42 (lactancia / IMC)

In [ ]:
endes_raw = pd.read_csv("datasets/endes_2024/REC42_2024.csv", encoding="utf-8-sig", low_memory=False)
print("ANTES:", endes_raw.shape)

endes = endes_raw.rename(columns={
    "V401": "tuvo_hijos_alguna_vez",
    "V404": "lacta_actualmente",
    "V437": "peso_kg_x10",
    "V438": "talla_cm_x10",
    "V445": "imc_x100",
})
for c in ["imc_x100", "peso_kg_x10", "talla_cm_x10"]:
    if c in endes.columns:
        endes.loc[endes[c].isin([9998, 9999]), c] = np.nan
endes["imc"] = endes["imc_x100"] / 100
endes["peso_kg"] = endes["peso_kg_x10"] / 10
endes["talla_cm"] = endes["talla_cm_x10"] / 10
q1, q3 = endes["imc"].quantile([0.25, 0.75])
iqr = q3 - q1
endes["imc_outlier"] = ~endes["imc"].between(q1 - 1.5*iqr, q3 + 1.5*iqr)
endes = endes[["ID1", "CASEID", "tuvo_hijos_alguna_vez", "lacta_actualmente",
               "imc", "peso_kg", "talla_cm", "imc_outlier"]]

print("DESPUES:", endes.shape)
print("IMC media:", round(endes['imc'].mean(), 2))
endes.head(3)

## 4. ENAHO 2024 - Modulo 01

In [ ]:
enaho_raw = pd.read_csv("datasets/enaho_2024/Enaho01-2024-100.csv", encoding="latin-1", low_memory=False)
print("ANTES:", enaho_raw.shape)

enaho = enaho_raw.copy()
enaho.columns = [quitar_tildes(c).replace(" ", "_") for c in enaho.columns]
enaho["UBIGEO"] = enaho["UBIGEO"].apply(lambda x: str(int(x)).zfill(6))
mapa_dom = {1: "COSTA NORTE", 2: "COSTA CENTRO", 3: "COSTA SUR",
            4: "SIERRA NORTE", 5: "SIERRA CENTRO", 6: "SIERRA SUR",
            7: "SELVA", 8: "LIMA METROPOLITANA"}
enaho["DOMINIO_NOMBRE"] = enaho["DOMINIO"].map(mapa_dom)
nunique = enaho.nunique(dropna=True)
enaho = enaho.loc[:, nunique > 1]
umbral = int(0.2 * len(enaho))
enaho = enaho.dropna(axis=1, thresh=umbral)

print("DESPUES:", enaho.shape)
enaho[["ANO", "MES", "UBIGEO", "DOMINIO", "DOMINIO_NOMBRE"]].head()

## 5. NNA atendidos en CEDIF

In [ ]:
import glob

esquema = ["COD_USU", "SEX_USU", "FEC_NAC_USU", "EDAD_USU", "GRU_ET",
           "PAI_USU", "TIE_DIS", "LEN_MAT", "AUT_IDE_ET", "NOM_CEN",
           "FEC_ING", "PER_ING", "TIP_SEG_SAL", "EST_ACT", "FEC_EGR", "MOT_EGR"]

trozos = []
for f in sorted(glob.glob("datasets/cuidado_diurno_nna/*.csv")):
    d = pd.read_csv(f, sep=";", encoding="latin-1", low_memory=False)
    if all(c in d.columns for c in esquema):
        d["__archivo__"] = f.split("/")[-1]
        trozos.append(d[esquema + ["__archivo__"]])

nna_raw = pd.concat([pd.read_csv(f, sep=';', encoding='latin-1', low_memory=False) for f in sorted(glob.glob('datasets/cuidado_diurno_nna/*.csv'))], ignore_index=True)
print("ANTES (concatenado bruto):", nna_raw.shape, "| dups:", nna_raw.duplicated().sum())

nna = pd.concat(trozos, ignore_index=True).drop_duplicates()
nna["PAI_USU"] = nna["PAI_USU"].apply(quitar_tildes)
for c in ["FEC_NAC_USU", "FEC_ING", "FEC_EGR"]:
    nna[c] = pd.to_datetime(nna[c], errors="coerce")
nna["SEXO"] = nna["SEX_USU"].map({1: "MASCULINO", 2: "FEMENINO"})
nna["EDAD_USU"] = pd.to_numeric(nna["EDAD_USU"], errors="coerce")
nna = nna[nna["EDAD_USU"].between(0, 17)]
nna["EDAD_USU"] = nna["EDAD_USU"].astype("int8")
for c in ["NOM_CEN", "PAI_USU", "SEXO"]:
    nna[c] = nna[c].astype("category")

print("DESPUES:", nna.shape, "|", round(nna.memory_usage(deep=True).sum()/1024/1024, 1), "MB")
nna.head(3)

## Tabla comparativa final

In [ ]:
resumen = pd.DataFrame([
    {"dataset": "ETES 2024", "filas_a": etes_raw.shape[0], "filas_d": etes.shape[0], "cols_a": etes_raw.shape[1], "cols_d": etes.shape[1]},
    {"dataset": "Fallecidos COVID", "filas_a": fall_raw.shape[0], "filas_d": fall.shape[0], "cols_a": fall_raw.shape[1], "cols_d": fall.shape[1]},
    {"dataset": "ENDES 2024", "filas_a": endes_raw.shape[0], "filas_d": endes.shape[0], "cols_a": endes_raw.shape[1], "cols_d": endes.shape[1]},
    {"dataset": "ENAHO 2024", "filas_a": enaho_raw.shape[0], "filas_d": enaho.shape[0], "cols_a": enaho_raw.shape[1], "cols_d": enaho.shape[1]},
    {"dataset": "NNA CEDIF", "filas_a": nna_raw.shape[0], "filas_d": nna.shape[0], "cols_a": nna_raw.shape[1], "cols_d": nna.shape[1]},
])
resumen